# Regression feasibility checkpoint
UCI Communities and Crime | OLS, Ridge, Lasso. Run all cells in Google Colab. Results are generated from actual data, not fabricated.

In [1]:
import numpy as np, pandas as pd, sklearn
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, r2_score
print("scikit-learn", sklearn.__version__)

scikit-learn 1.9.0


## Load UCI dataset
The `.names` file defines the attributes. `?` denotes missing values. Dataset is fetched when this notebook runs online.

In [2]:
base="https://archive.ics.uci.edu/ml/machine-learning-databases/communities/"
import urllib.request
names=urllib.request.urlopen(base+"communities.names").read().decode("latin-1")
cols=[line.split()[1] for line in names.splitlines() if line.startswith("@attribute ")]
df=pd.read_csv(base+"communities.data",header=None,names=cols,na_values="?")
print("Shape:",df.shape,"Target missing:",df["ViolentCrimesPerPop"].isna().sum())
display(df.head(3))

Shape: (1994, 128) Target missing: 0


,state,county,community,communityname,fold,population,householdsize,racepctblack,racePctWhite,racePctAsian,...,LandArea,PopDens,PctUsePubTrans,PolicCars,PolicOperBudg,LemasPctPolicOnPatr,LemasGangUnitDeploy,LemasPctOfficDrugUn,PolicBudgPerPop,ViolentCrimesPerPop
0,8,NaN,NaN,Lakewoodcity,1,0.19,0.33,0.02,0.90,0.12,...,0.12,0.26,0.20,0.06,0.04,0.9,0.5,0.32,0.14,0.20
1,53,NaN,NaN,Tukwilacity,1,0.00,0.16,0.12,0.74,0.45,...,0.02,0.12,0.45,NaN,NaN,NaN,NaN,0.00,NaN,0.67
2,24,NaN,NaN,Aberdeentown,1,0.00,0.42,0.49,0.56,0.17,...,0.01,0.21,0.02,NaN,NaN,NaN,NaN,0.00,NaN,0.43


## Prepare identical train/validation/test partitions
The held-out test partition is evaluated only after model selection. Missing-value imputation and scaling are learned inside training folds.

In [3]:
target="ViolentCrimesPerPop"
exclude=["state","county","community","communityname","fold",target]
X=df.drop(columns=[c for c in exclude if c in df.columns]).apply(pd.to_numeric,errors="coerce")
y=pd.to_numeric(df[target],errors="coerce")
valid=y.notna(); X=X.loc[valid]; y=y.loc[valid]
# Drop columns with more than 50% missing based on training set only.
X_trainval,X_test,y_trainval,y_test=train_test_split(X,y,test_size=.2,random_state=42)
X_train,X_val,y_train,y_val=train_test_split(X_trainval,y_trainval,test_size=.25,random_state=42)
keep=X_train.columns[X_train.isna().mean()<=.5].tolist()
X_train=X_train[keep];X_val=X_val[keep];X_test=X_test[keep]
print("Train/validation/test:",len(X_train),len(X_val),len(X_test),"Features:",len(keep))

Train/validation/test: 1196 399 399 Features: 100


## Initial feasibility baseline and tuned Ridge/Lasso
Five-fold cross-validation on training data only. Validation metrics below are genuine when executed.

In [4]:
def pipeline(model):
 return Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler()),("model",model)])
models={"Mean baseline":pipeline(DummyRegressor(strategy="mean")),"OLS":pipeline(LinearRegression())}
cv=KFold(n_splits=5,shuffle=True,random_state=42)
for label,est in [("Ridge",Ridge()),("Lasso",Lasso(max_iter=20000))]:
 search=GridSearchCV(pipeline(est),{"model__alpha":np.logspace(-4,3,20)},scoring="neg_root_mean_squared_error",cv=cv,n_jobs=-1)
 search.fit(X_train,y_train);models[label]=search.best_estimator_
 print(label,"best alpha:",search.best_params_["model__alpha"])
rows=[]
for label,model in models.items():
 model.fit(X_train,y_train);pred=model.predict(X_val)
 rows.append({"Model":label,"Validation RMSE":np.sqrt(mean_squared_error(y_val,pred)),"Validation R2":r2_score(y_val,pred)})
results=pd.DataFrame(rows).sort_values("Validation RMSE")
display(results.round(5))
results.to_csv("feasibility_results.csv",index=False)

Ridge best alpha: 33.59818286283781
Lasso best alpha: 0.000545559478116852


,Model,Validation RMSE,Validation R2
2,Ridge,0.14267,0.61791
3,Lasso,0.14326,0.61473
1,OLS,0.14629,0.59830
0,Mean baseline,0.23086,-0.00048


## Next steps
Expand the alpha grid to 50 values, evaluate 20 fixed seeds, examine training-size effects and coefficient stability with bootstrap resampling. After selection, report held-out test results. Document versions and runtime.